In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

# =========================
# 0) Config
# =========================
CSV_PATH = "cleaned_trade_master.csv"
YEARS = [2023, 2024, 2025, 2026, 2027, 2028]
OUT_TOP5_IMPORT  = "top5_products_per_country_IMPORT.csv"
OUT_TOP5_EXPORT  = "top5_products_per_country_EXPORT.csv"
OUT_MODEL_METRICS = "model_metrics.txt"

# =========================
# 1) Load + clean
# =========================
df = pd.read_csv(CSV_PATH)

# remove duplicate-named columns (keep first)
df = df.loc[:, ~df.columns.duplicated()].copy()

# keep rows with target
df = df.dropna(subset=["Value_2023"]).copy()
df["Value_2023"] = pd.to_numeric(df["Value_2023"], errors="coerce")

# normalize text fields
for col in ["Country", "Product"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

if "Direction" in df.columns:
    df["Direction"] = (df["Direction"].astype(str)
                       .str.strip().str.lower()
                       .replace({"imp":"import","exp":"export"})
                       .str.capitalize())

if "Code" in df.columns:
    df["Code"] = df["Code"].astype(str).str.strip()

# coerce growth-like columns to numeric
for gcol in ["World_Growth", "Growth_2022_2023", "Growth_2019_2023"]:
    if gcol in df.columns:
        df[gcol] = pd.to_numeric(df[gcol], errors="coerce")

# =========================
# 2) Features / target
# =========================
# numeric candidates (use only those that exist)
desired_num = [
    "Trade_Balance_2023","Growth_2019_2023","Growth_2022_2023",
    "World_Growth","World_Import_Rank","Avg_Distance_km",
    "Concentration","World_Import_Share","Avg_Tariff"
]
desired_cat = ["Country","Direction","Code"]  # include product code as categorical

num_cols = [c for c in desired_num if c in df.columns]
cat_cols = [c for c in desired_cat if c in df.columns]

X = df[num_cols + cat_cols].copy()
y = df["Value_2023"].astype(float)

# =========================
# 3) Preprocess + split
# =========================
preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline(steps=[
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 4) Train models
# =========================
lr = Pipeline(steps=[("prep", preprocess), ("m", LinearRegression())])
xgb = Pipeline(steps=[("prep", preprocess), ("m", XGBRegressor(
    n_estimators=450, learning_rate=0.07, max_depth=6,
    subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=2
))])

lr.fit(X_train, y_train)
xgb.fit(X_train, y_train)

def eval_model(name, model, Xte, yte):
    pred = model.predict(Xte)
    r2   = r2_score(yte, pred)
    rmse = mean_squared_error(yte, pred, squared=False)
    mae  = mean_absolute_error(yte, pred)
    print(f"\n{name}\nR²: {r2:.4f} | RMSE: {rmse:,.0f} | MAE: {mae:,.0f}")
    return r2, rmse, mae

r2_lr,  rmse_lr,  mae_lr  = eval_model("Linear Regression", lr,  X_test, y_test)
r2_xgb, rmse_xgb, mae_xgb = eval_model("XGBoost Regressor", xgb, X_test, y_test)

with open(OUT_MODEL_METRICS, "w") as f:
    f.write(f"Linear Regression -> R2={r2_lr:.4f}, RMSE={rmse_lr:.0f}, MAE={mae_lr:.0f}\n")
    f.write(f"XGBoost Regressor -> R2={r2_xgb:.4f}, RMSE={rmse_xgb:.0f}, MAE={mae_xgb:.0f}\n")

# =========================
# 5) Predict base 2023 + build per-row growth rate
# =========================
df["Pred_2023_XGB"] = xgb.predict(X)

# robust per-row growth rate:
# prefer recent product-specific growth; blend with world growth; fallback to 0
g_recent = df["Growth_2022_2023"] / 100.0 if "Growth_2022_2023" in df.columns else np.nan
g_world  = df["World_Growth"]       / 100.0 if "World_Growth" in df.columns else np.nan

g = np.where(~pd.isna(g_recent) & ~pd.isna(g_world), 0.6*g_recent + 0.4*g_world,
     np.where(~pd.isna(g_recent), g_recent,
     np.where(~pd.isna(g_world),  g_world, 0.0)))

g = pd.Series(g, index=df.index)

# =========================
# 6) Forecast to 2028 (recursive compounding)
# =========================
def compound_series(base, gr, years_out):
    """Return dict year->values per row (includes 2023 base)."""
    out = {}
    val = base.values.astype(float).copy()
    out[2023] = val.copy()
    for y in years_out[1:]:
        val = val * (1.0 + gr.values)
        out[y] = val.copy()
    return out

track_xgb = compound_series(df["Pred_2023_XGB"], g, YEARS)

# Build a long forecast table row-by-row (to aggregate later)
rows = []
for y in YEARS:
    rows.append(pd.DataFrame({
        "Country":  df["Country"],
        "Product":  df["Product"] if "Product" in df.columns else "",
        "Code":     df["Code"] if "Code" in df.columns else "",
        "Direction":df["Direction"] if "Direction" in df.columns else "",
        "Year":     y,
        "Forecast_Value": track_xgb[y]
    }))
forecast_rows = pd.concat(rows, ignore_index=True)

# =========================
# 7) Rank Top-5 products per country (Import / Export separately)
#     — by CAGR 2023→2028; tie-break on 2028 size
# =========================
# 2023 base by Country x Product x Direction (use actual if present)
base_2023 = (df.groupby(["Country","Product","Code","Direction"], as_index=False)["Value_2023"]
               .sum().rename(columns={"Value_2023":"Base_2023"}))

f_2028 = (forecast_rows[forecast_rows["Year"]==2028]
          .groupby(["Country","Product","Code","Direction"], as_index=False)["Forecast_Value"].sum()
          .rename(columns={"Forecast_Value":"F_2028"}))

prod_growth = base_2023.merge(f_2028, on=["Country","Product","Code","Direction"], how="outer")
prod_growth["Pct_Growth_2023_2028"] = np.where(
    prod_growth["Base_2023"]>0,
    (prod_growth["F_2028"]/prod_growth["Base_2023"] - 1.0)*100.0,
    np.nan
)
prod_growth["CAGR_2023_2028_%"] = np.where(
    prod_growth["Base_2023"]>0,
    ((prod_growth["F_2028"]/prod_growth["Base_2023"])**(1/5) - 1.0)*100.0,
    np.nan
)

# helper to pick top 5 per country for a given direction
def top5_per_country(df_in, direction):
    df_dir = df_in[df_in["Direction"].eq(direction)].copy()
    # rank by CAGR then by absolute 2028 value to break ties
    df_dir["__rankkey__"] = list(zip(-df_dir["CAGR_2023_2028_%"].fillna(-1e9),
                                     -df_dir["F_2028"].fillna(-1e9)))
    # sort within country then take head(5)
    out = (df_dir.sort_values(["Country","__rankkey__"])
                 .groupby("Country", as_index=False)
                 .head(5)
                 .drop(columns="__rankkey__"))
    # tidy columns
    cols = ["Country","Product","Code","Base_2023","F_2028","Pct_Growth_2023_2028","CAGR_2023_2028_%","Direction"]
    return out[cols]

top5_import  = top5_per_country(prod_growth, "Import")
top5_export  = top5_per_country(prod_growth, "Export")

top5_import.to_csv(OUT_TOP5_IMPORT, index=False)
top5_export.to_csv(OUT_TOP5_EXPORT, index=False)

print(f"\nSaved: {OUT_TOP5_IMPORT}")
print(f"Saved: {OUT_TOP5_EXPORT}")

# Optional: show a couple examples quickly in console
def show_examples(country_code):
    print(f"\nTop-5 IMPORT products for {country_code}")
    print(top5_import[top5_import["Country"]==country_code]
          .sort_values("CAGR_2023_2028_%", ascending=False)
          .to_string(index=False))
    print(f"\nTop-5 EXPORT products for {country_code}")
    print(top5_export[top5_export["Country"]==country_code]
          .sort_values("CAGR_2023_2028_%", ascending=False)
          .to_string(index=False))

# e.g., show_examples("MAR")   # uncomment to preview Morocco



Linear Regression
R²: 0.0146 | RMSE: 4,282,024 | MAE: 1,096,792

XGBoost Regressor
R²: 0.8438 | RMSE: 1,704,639 | MAE: 230,060

Saved: top5_products_per_country_IMPORT.csv
Saved: top5_products_per_country_EXPORT.csv


In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

# =========================
# Config
# =========================
CSV_PATH = "cleaned_trade_master.csv"  # <- change if needed

OUT_METRICS            = "growth_model_metrics.txt"
OUT_TOP5_IMPORT_CAGR   = "top5_growth_products_per_country_IMPORT.csv"
OUT_TOP5_EXPORT_CAGR   = "top5_growth_products_per_country_EXPORT.csv"
OUT_TOP5_IMPORT_SCORE  = "top5_potential_products_per_country_IMPORT.csv"
OUT_TOP5_EXPORT_SCORE  = "top5_potential_products_per_country_EXPORT.csv"

# =========================
# 1) Load + clean
# =========================
df = pd.read_csv(CSV_PATH)
df = df.loc[:, ~df.columns.duplicated()].copy()

# Normalize text fields
for col in ["Country","Product","Code"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

if "Direction" in df.columns:
    df["Direction"] = (
        df["Direction"].astype(str).str.strip().str.lower()
          .replace({"imp":"import","exp":"export","imports":"import","exports":"export"})
          .str.capitalize()
    )

# Coerce numeric columns that we’ll use
for c in ["Growth_2022_2023","Growth_2019_2023","World_Growth","World_Import_Rank",
          "Avg_Distance_km","Concentration","World_Import_Share","Avg_Tariff",
          "Trade_Balance_2023","Value_2023"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# =========================
# 2) Build a Growth Target (NOT value)
# =========================
# CAGR(2019->2023) from total growth %, approx over 4 years
cagr_19_23 = np.where(
    df["Growth_2019_2023"].notna(),
    (1.0 + df["Growth_2019_2023"]/100.0)**(1/4) - 1.0,
    np.nan
)  # decimal

g22_23 = df["Growth_2022_2023"]/100.0  # decimal
# Blend where both exist; fallback to whichever exists; else NaN
growth_target = np.where(~pd.isna(g22_23) & ~pd.isna(cagr_19_23),
                         0.60*g22_23 + 0.40*cagr_19_23,
                         np.where(~pd.isna(g22_23), g22_23, cagr_19_23))
df["Growth_Target"] = growth_target * 100.0  # store as percent points

# Drop rows without target
df = df.dropna(subset=["Growth_Target"]).copy()

# =========================
# 3) Features / preprocessing
# =========================
desired_num = [
    "World_Growth","World_Import_Rank","Avg_Distance_km",
    "Concentration","World_Import_Share","Avg_Tariff",
    "Trade_Balance_2023"
]
desired_cat = ["Country","Direction","Code"]  # include HS code signal

num_cols = [c for c in desired_num if c in df.columns]
cat_cols = [c for c in desired_cat if c in df.columns]

X = df[num_cols + cat_cols].copy()
y = df["Growth_Target"].astype(float)  # % growth target

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline(steps=[
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ]
)

# =========================
# 4) Train models (predict growth)
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lr = Pipeline(steps=[("prep", preprocess), ("m", LinearRegression())])
xgb = Pipeline(steps=[("prep", preprocess), ("m", XGBRegressor(
    n_estimators=450, learning_rate=0.07, max_depth=6,
    subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=2
))])

lr.fit(X_train, y_train)
xgb.fit(X_train, y_train)

def eval_model(name, model, Xte, yte):
    pred = model.predict(Xte)
    r2   = r2_score(yte, pred)
    rmse = mean_squared_error(yte, pred, squared=False)
    mae  = mean_absolute_error(yte, pred)
    print(f"\n{name}  |  R²={r2:.3f}  RMSE={rmse:.2f}  MAE={mae:.2f}")
    return r2, rmse, mae

r2_lr, rmse_lr, mae_lr   = eval_model("Linear Regression",  lr,  X_test, y_test)
r2_xgb, rmse_xgb, mae_xgb= eval_model("XGBoost Regressor", xgb, X_test, y_test)

with open(OUT_METRICS, "w") as f:
    f.write(f"Linear Regression  -> R2={r2_lr:.4f}, RMSE={rmse_lr:.2f}, MAE={mae_lr:.2f}\n")
    f.write(f"XGBoost Regressor -> R2={r2_xgb:.4f}, RMSE={rmse_xgb:.2f}, MAE={mae_xgb:.2f}\n")

# =========================
# 5) Predict growth for all rows
# =========================
df["Pred_Growth_LR_%"]  = lr.predict(X)     # % points
df["Pred_Growth_XGB_%"] = xgb.predict(X)    # % points

# =========================
# 6) Optional: Composite “Potential Score”
#     (combine growth with practical frictions / signals)
#     All terms are standardized to be comparable; weights chosen to
#     favor growth but penalize distance/tariff/concentration.
# =========================
def z(x):
    x = pd.to_numeric(x, errors="coerce")
    return (x - x.mean()) / (x.std(ddof=0) + 1e-9)

z_pred_g = z(df["Pred_Growth_XGB_%"])
z_world  = z(df["World_Growth"]) if "World_Growth" in df else 0
z_share  = z(df["World_Import_Share"]) if "World_Import_Share" in df else 0
z_rank   = -z(df["World_Import_Rank"]) if "World_Import_Rank" in df else 0  # lower rank is better
z_tariff = -z(df["Avg_Tariff"]) if "Avg_Tariff" in df else 0               # lower tariff better
z_dist   = -z(df["Avg_Distance_km"]) if "Avg_Distance_km" in df else 0     # closer better
z_conc   = -z(df["Concentration"]) if "Concentration" in df else 0         # less concentrated better

df["Potential_Score"] = (
    0.70 * z_pred_g +
    0.15 * z_world   +
    0.05 * z_share   +
    0.04 * z_rank    +
    0.03 * z_tariff  +
    0.02 * z_dist    +
    0.01 * z_conc
)

# =========================
# 7) Rank Top-5 products per country (Import/Export)
#     A) by Predicted Growth (CAGR proxy)
#     B) by Composite Potential Score
# =========================
def topk_per_country(df_in, direction, by_col, k=5):
    d = df_in[df_in["Direction"].eq(direction)].copy()
    # tie-breaker: prefer larger world share and better rank where available
    if "World_Import_Share" in d.columns:
        d["__tieb__"] = d["World_Import_Share"].fillna(0)
    else:
        d["__tieb__"] = 0.0
    d = d.sort_values(["Country", by_col, "__tieb__"], ascending=[True, False, False])
    out = d.groupby("Country", as_index=False).head(k).drop(columns="__tieb__")
    keep = ["Country","Direction","Product","Code","Pred_Growth_XGB_%","Potential_Score",
            "World_Growth","World_Import_Share","World_Import_Rank","Avg_Tariff","Avg_Distance_km","Concentration"]
    keep = [c for c in keep if c in out.columns]
    return out[keep]

# A) Top-5 by predicted growth
top5_imp_cagr = topk_per_country(df, "Import", "Pred_Growth_XGB_%", k=5)
top5_exp_cagr = topk_per_country(df, "Export", "Pred_Growth_XGB_%", k=5)
top5_imp_cagr.to_csv(OUT_TOP5_IMPORT_CAGR, index=False)
top5_exp_cagr.to_csv(OUT_TOP5_EXPORT_CAGR, index=False)

# B) Top-5 by potential score
top5_imp_score = topk_per_country(df, "Import", "Potential_Score", k=5)
top5_exp_score = topk_per_country(df, "Export", "Potential_Score", k=5)
top5_imp_score.to_csv(OUT_TOP5_IMPORT_SCORE, index=False)
top5_exp_score.to_csv(OUT_TOP5_EXPORT_SCORE, index=False)

print("\nSaved:")
print(" -", OUT_METRICS)
print(" -", OUT_TOP5_IMPORT_CAGR)
print(" -", OUT_TOP5_EXPORT_CAGR)
print(" -", OUT_TOP5_IMPORT_SCORE)
print(" -", OUT_TOP5_EXPORT_SCORE)



Linear Regression  |  R²=-0.047  RMSE=60.92  MAE=30.71

XGBoost Regressor  |  R²=-0.741  RMSE=78.56  MAE=29.04

Saved:
 - growth_model_metrics.txt
 - top5_growth_products_per_country_IMPORT.csv
 - top5_growth_products_per_country_EXPORT.csv
 - top5_potential_products_per_country_IMPORT.csv
 - top5_potential_products_per_country_EXPORT.csv


In [11]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

# =========================
# Config
# =========================
CSV_PATH = "cleaned_trade_master.csv"   # <- update if needed

OUT_METRICS             = "growth_model_metrics_NO_TARIFF_SHARE.txt"
OUT_TOP5_IMPORT_GROWTH  = "top5_products_per_country_IMPORT_growth_ONLY.csv"
OUT_TOP5_EXPORT_GROWTH  = "top5_products_per_country_EXPORT_growth_ONLY.csv"

# =========================
# 1) Load + basic cleaning
# =========================
df = pd.read_csv(CSV_PATH)

# remove duplicate-named columns (keep first)
df = df.loc[:, ~df.columns.duplicated()].copy()

# normalize text fields
for col in ["Country", "Product", "Code"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

if "Direction" in df.columns:
    df["Direction"] = (
        df["Direction"].astype(str)
        .str.strip().str.lower()
        .replace({"imp": "import", "exp": "export", "imports": "import", "exports": "export"})
        .str.capitalize()
    )

# coerce numerics we might use
for c in [
    "Growth_2022_2023", "Growth_2019_2023", "World_Growth",
    "World_Import_Rank", "Avg_Distance_km", "Concentration",
    "Trade_Balance_2023", "Value_2023"
]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# =========================
# 2) Build a Growth Target (NOT trade value)
#    Target = 60% recent growth + 40% CAGR(2019->2023)
# =========================
# recent one-year growth (decimal)
g22_23 = df["Growth_2022_2023"] / 100.0 if "Growth_2022_2023" in df.columns else np.nan

# approx CAGR across 2019->2023 (4 years) from cumulative % growth
if "Growth_2019_2023" in df.columns:
    cagr_19_23 = (1.0 + (df["Growth_2019_2023"] / 100.0)) ** (1/4) - 1.0
else:
    cagr_19_23 = np.nan

growth_target = np.where(
    (~pd.isna(g22_23)) & (~pd.isna(cagr_19_23)),
    0.60 * g22_23 + 0.40 * cagr_19_23,
    np.where(~pd.isna(g22_23), g22_23, cagr_19_23)
)
df["Growth_Target_%"] = growth_target * 100.0  # store as %-points

# drop rows without a target
df = df.dropna(subset=["Growth_Target_%"]).copy()

# =========================
# 3) Features (EXCLUDING tariff & world import share, and no potential score)
# =========================
desired_num = [
    "World_Growth", "World_Import_Rank", "Avg_Distance_km",
    "Concentration", "Trade_Balance_2023",
    "Growth_2022_2023", "Growth_2019_2023"
]
desired_cat = ["Country", "Direction", "Code"]  # include HS code signal (optional)

num_cols = [c for c in desired_num if c in df.columns]
cat_cols = [c for c in desired_cat if c in df.columns]

X = df[num_cols + cat_cols].copy()
y = df["Growth_Target_%"].astype(float)

# =========================
# 4) Preprocess + split
# =========================
preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline(steps=[
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# =========================
# 5) Train models to predict growth
# =========================
lr = Pipeline(steps=[("prep", preprocess), ("m", LinearRegression())])

xgb = Pipeline(steps=[("prep", preprocess), ("m", XGBRegressor(
    n_estimators=450, learning_rate=0.07, max_depth=6,
    subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=2
))])

lr.fit(X_train, y_train)
xgb.fit(X_train, y_train)

def eval_model(name, model, Xte, yte):
    pred = model.predict(Xte)
    r2   = r2_score(yte, pred)
    rmse = mean_squared_error(yte, pred, squared=False)
    mae  = mean_absolute_error(yte, pred)
    print(f"\n{name}  |  R²={r2:.3f}  RMSE={rmse:.2f}  MAE={mae:.2f}")
    return r2, rmse, mae

r2_lr, rmse_lr, mae_lr     = eval_model("Linear Regression",  lr,  X_test, y_test)
r2_xgb, rmse_xgb, mae_xgb  = eval_model("XGBoost Regressor", xgb, X_test, y_test)

with open(OUT_METRICS, "w") as f:
    f.write(f"Linear Regression  -> R2={r2_lr:.4f}, RMSE={rmse_lr:.2f}, MAE={mae_lr:.2f}\n")
    f.write(f"XGBoost Regressor -> R2={r2_xgb:.4f}, RMSE={rmse_xgb:.2f}, MAE={mae_xgb:.2f}\n")

# =========================
# 6) Predict growth for ALL rows
# =========================
df["Pred_Growth_XGB_%"] = xgb.predict(X)  # our ranking signal (higher = faster growth)

# =========================
# 7) Rank Top-5 products per country
#     Separately for Import and Export
#     (No tariff/share; pure predicted growth)
# =========================
def topk_per_country_growth(df_in, direction, k=5):
    if "Direction" not in df_in.columns:
        # if Direction absent, treat all as same group
        d = df_in.copy()
    else:
        d = df_in[df_in["Direction"].eq(direction)].copy()
    if d.empty:
        return pd.DataFrame(columns=["Country","Direction","Product","Code","Pred_Growth_XGB_%"])

    # optional tie-break: prefer higher World_Growth if available
    if "World_Growth" in d.columns:
        d["__tieb__"] = d["World_Growth"].fillna(0)
    else:
        d["__tieb__"] = 0.0

    d = d.sort_values(["Country", "Pred_Growth_XGB_%", "__tieb__"],
                      ascending=[True, False, False])

    out = d.groupby("Country", as_index=False).head(k).drop(columns="__tieb__")
    keep = ["Country", "Direction", "Product", "Code", "Pred_Growth_XGB_%",
            "World_Growth", "World_Import_Rank", "Avg_Distance_km", "Concentration",
            "Trade_Balance_2023", "Growth_2022_2023", "Growth_2019_2023"]
    keep = [c for c in keep if c in out.columns]
    return out[keep]

top5_import_growth = topk_per_country_growth(df, "Import", k=5)
top5_export_growth = topk_per_country_growth(df, "Export", k=5)

top5_import_growth.to_csv(OUT_TOP5_IMPORT_GROWTH, index=False)
top5_export_growth.to_csv(OUT_TOP5_EXPORT_GROWTH, index=False)

print("\nSaved:")
print(" -", OUT_METRICS)
print(" -", OUT_TOP5_IMPORT_GROWTH)
print(" -", OUT_TOP5_EXPORT_GROWTH)



Linear Regression  |  R²=0.999  RMSE=2.06  MAE=1.26

XGBoost Regressor  |  R²=0.976  RMSE=9.25  MAE=1.74

Saved:
 - growth_model_metrics_NO_TARIFF_SHARE.txt
 - top5_products_per_country_IMPORT_growth_ONLY.csv
 - top5_products_per_country_EXPORT_growth_ONLY.csv
